# Shared Stage 1 model evaluation: training, validation, and test

Owner: Chua Rui Ning, Renee

This notebook evaluates training, validation, and test in separate runs. Each split uses its own labels and model rankings, and writes reports only to its matching folder:

- Training: Evaluation/Stage 1/training
- Validation: Evaluation/Stage 1/validation
- Test: Evaluation/Stage 1/test

Training metrics are in-sample; validation is for model selection and reranker tuning. Test is for final reporting after choices are frozen. Training, validation, and test use separate labels and outputs.

In [7]:
%pip -q install numpy pandas pyarrow


In [8]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configure split inputs and outputs

Set the paths in the next cell to the exact full-split top-200 rankings. Each split's rankings are paired only with its own labels. Training output is in-sample; validation supports selection; test is final evaluation. Outputs are kept in separate split folders. Training evaluates LSTM, XGBoost, and Gemini; RRF is evaluated on validation and test because no train RRF export is configured. Copy the updated `evaluate.py` into Drive's `Evaluation/code/` so training reports are labelled `in_sample_training`.

In [9]:
from pathlib import Path
import json
import io
import sys
import warnings
from contextlib import redirect_stdout

import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder')
EVALUATION_ROOT = PROJECT_ROOT / 'Evaluation'
STAGE1_OUTPUT_ROOT = EVALUATION_ROOT / 'Stage 1'

TRAIN_VAL_ARTICLES_PATH = PROJECT_ROOT / 'train_val' / 'articles.parquet'
TEST_ARTICLES_CANDIDATE = PROJECT_ROOT / 'test' / 'articles (1).parquet'
TEST_ARTICLES_PATH = (
    TEST_ARTICLES_CANDIDATE
    if TEST_ARTICLES_CANDIDATE.is_file()
    else PROJECT_ROOT / 'test' / 'articles.parquet'
)
ARTICLES_PATH = EVALUATION_ROOT / 'catalogue' / 'article_union.parquet'

LABELS_BY_SPLIT = {
    'train': PROJECT_ROOT / 'feature_engineering' / 'train_features.parquet',
    'validation': PROJECT_ROOT / 'feature_engineering' / 'validation_features.parquet',
    'test': PROJECT_ROOT / 'feature_engineering' / 'test_features.parquet',
}

EVALUATION_CODE_DIR = EVALUATION_ROOT / 'code'
required_modules = ('evaluate.py', 'build_catalogue_union.py')
missing_modules = [
    name for name in required_modules
    if not (EVALUATION_CODE_DIR / name).is_file()
]
if missing_modules:
    raise FileNotFoundError(
        f'Missing {missing_modules} under {EVALUATION_CODE_DIR}. '
        'Copy the evaluator Python files into Drive at Evaluation/code/.'
    )
sys.path.insert(0, str(EVALUATION_CODE_DIR))


def existing_file(path):
    path = Path(path)
    return path if path.is_file() else None


LSTM_RUN_DIR = (
    PROJECT_ROOT / 'Experiments' / '2.1_lstm' / '2.1.3_improved_run'
    / '20261104_175920_improved_seed42'
)
XGBOOST_RUN_DIR = (
    PROJECT_ROOT / 'Experiments' / '1.3_xgboost' / '20261105_small_seed42'
)
GEMINI_RUN_DIR = (
    PROJECT_ROOT / 'Experiments'
    / '3.1_pretrained_similarity_gemini_individual_history'
    / '20261108_history_seed42'
)
RRF_ROOT = EVALUATION_ROOT / 'ensemble'

RUN_DEFINITIONS = [
    {
        'name': 'LSTM',
        'run_dir': LSTM_RUN_DIR,
        'catalogue_scores': True,
        'recorded_slate_scores': True,
    },
    {
        'name': 'XGBoost',
        'run_dir': XGBOOST_RUN_DIR,
        'catalogue_scores': True,
        'recorded_slate_scores': False,
    },
    {
        'name': 'Gemini',
        'run_dir': GEMINI_RUN_DIR,
        'catalogue_scores': True,
        'recorded_slate_scores': False,
    },
    {
        'name': 'RRF_lstm75_xgboost25',
        'rankings_by_split': {
            'validation': (
                RRF_ROOT / 'validation'
                / 'rrf_lstm75_xgboost25_validation_top_200.parquet'
            ),
            'test': (
                RRF_ROOT / 'test'
                / 'rrf_lstm75_xgboost25_test_top_200.parquet'
            ),
        },
        'catalogue_scores': False,
        'recorded_slate_scores': False,
    },
]


def model_runs_for_split(split):
    runs = []
    for definition in RUN_DEFINITIONS:
        if 'rankings_by_split' in definition:
            if split not in definition['rankings_by_split']:
                continue
            ranking_path = definition['rankings_by_split'][split]
            run_dir = None
        else:
            run_dir = definition['run_dir']
            ranking_path = (
                run_dir / 'rankings' / split / 'submission_top_200.parquet'
            )

        run_config = None
        catalogue_scores = None
        recorded_slate_scores = None
        if run_dir is not None:
            # Attach only a config explicitly scoped to this split.
            config_name = {
                'train': 'run_config_train.json',
                'validation': 'run_config.json',
                'test': 'run_config_test.json',
            }[split]
            run_config = existing_file(run_dir / config_name)

            if definition['catalogue_scores']:
                catalogue_scores = existing_file(
                    run_dir / 'predictions' / split
                    / 'submission_article_scores.parquet'
                )
            if definition['recorded_slate_scores']:
                recorded_slate_scores = existing_file(
                    run_dir / 'diagnostics' / 'recorded_slates'
                    / f'{split}_scores.parquet'
                )

        runs.append({
            'name': definition['name'],
            'rankings': ranking_path,
            'run_config': run_config,
            'catalogue_scores': catalogue_scores,
            'recorded_slate_scores': recorded_slate_scores,
        })
    return runs


SPLITS = ('train', 'validation', 'test')
SPLIT_OUTPUT_NAMES = {'train': 'training', 'validation': 'validation', 'test': 'test'}
EXPORT_K = 200
KS = [100, 200]
ALLOW_SUBSET = False
NAIVE_TIMEZONE = 'UTC'

print('Project root exists:', PROJECT_ROOT.exists())
print('Catalogue inputs:', TRAIN_VAL_ARTICLES_PATH.is_file(), TEST_ARTICLES_PATH.is_file())
print('Labels by split:')
for split, path in LABELS_BY_SPLIT.items():
    print(f'  {split}: {path} ({path.is_file()})')
print('Evaluator code:', EVALUATION_CODE_DIR)
for split in SPLITS:
    print(f'{split} ranking paths:')
    for run in model_runs_for_split(split):
        print(f"  {run['name']}: {run['rankings']} ({Path(run['rankings']).is_file()})")

Project root exists: True
Catalogue inputs: True True
Labels by split:
  train: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/feature_engineering/train_features.parquet (True)
  validation: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/feature_engineering/validation_features.parquet (True)
  test: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/feature_engineering/test_features.parquet (True)
Evaluator code: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/code
train ranking paths:
  LSTM: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.1_lstm/2.1.3_improved_run/20261104_175920_improved_seed42/rankings/train/submission_top_200.parquet (True)
  XGBoost: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/1.3_xgboost/20261105_small_seed42/rankings/train/submission_top_200.parquet (True)
  Gemini: /content/drive/MyDrive/Colab Notebooks/BT4222 Shar

## Import shared evaluator implementation

The evaluator and catalogue builder are imported from Evaluation/code on Drive. The same evaluator code is applied independently to each split.

In [10]:
# Remove cached imports so this session uses the configured Drive modules.
for module_name in ('evaluate', 'similarity', 'build_catalogue_union'):
    sys.modules.pop(module_name, None)

from evaluate import main as evaluate_main
from build_catalogue_union import main as build_catalogue_union_main

## Build the shared article catalogue

The ordered article catalogue is shared across the evaluation runs. Rebuild it if either declared article snapshot changes. It uses train/validation precedence, then includes test-snapshot articles according to the project’s catalogue policy.

In [11]:
old_argv = sys.argv
try:
    sys.argv = [
        'build_catalogue_union.py',
        '--train-val', str(TRAIN_VAL_ARTICLES_PATH),
        '--test', str(TEST_ARTICLES_PATH),
        '--naive-timezone', NAIVE_TIMEZONE,
        '--output', str(ARTICLES_PATH),
    ]
    build_catalogue_union_main()
finally:
    sys.argv = old_argv

CATALOGUE_MANIFEST_PATH = ARTICLES_PATH.with_suffix(
    ARTICLES_PATH.suffix + '.manifest.json'
)
print('Catalogue:', ARTICLES_PATH)
print('Manifest:', CATALOGUE_MANIFEST_PATH)

{'train_val_rows': 20738, 'test_rows': 20738, 'union_rows': 20738, 'duplicate_ids_removed': 20738, 'overlap_ids': 20738, 'catalogue_id_fingerprint_sha256': 'eff069a2586ad6cd00273b1132999f38c98b1339424040d2af345c01121292c6', 'catalogue_content_fingerprint_sha256': 'bbd9dfd867eebdaef43598653d03af2ad3a8f4dc5a7d4ac11b210eb97d7e0728', 'text_flag_sources': [], 'image_flag_sources': [], 'output': '/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/catalogue/article_union.parquet', 'manifest': '/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/catalogue/article_union.parquet.manifest.json'}
Catalogue: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/catalogue/article_union.parquet
Manifest: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/catalogue/article_union.parquet.manifest.json


## Evaluate validation and test separately

Each split is evaluated only with its own labels and ranking exports. Reports and per-request files are written under that split’s Stage 1 folder. Test run configs are omitted unless a test-specific config exists; the evaluator still checks article eligibility and exact request coverage.

In [12]:
required_paths = [
    ARTICLES_PATH,
    TRAIN_VAL_ARTICLES_PATH,
    TEST_ARTICLES_PATH,
    *LABELS_BY_SPLIT.values(),
]
missing = [str(path) for path in required_paths if not Path(path).is_file()]
if missing:
    raise FileNotFoundError(
        'Required file(s) not found:\n' + '\n'.join(missing)
    )


def evaluator_compatible_run_config(run, split, output_dir):
    """Normalize supported catalogue-fingerprint aliases for one split."""
    source = run.get('run_config')
    if source is None:
        return None

    source = Path(source)
    if not source.is_file():
        raise FileNotFoundError(
            f"{run['name']} {split} run_config not found: {source}"
        )
    config = json.loads(source.read_text(encoding='utf-8'))
    eligibility = config.get('eligibility', {})
    needs_normalization = False

    if not config.get('catalogue_id_fingerprint_sha256'):
        shared_hash = eligibility.get('shared_catalogue_id_fingerprint_sha256')
        short_hash = eligibility.get('catalogue_fingerprint')
        if shared_hash:
            config['catalogue_id_fingerprint_sha256'] = shared_hash
            needs_normalization = True
        elif short_hash:
            if not CATALOGUE_MANIFEST_PATH.is_file():
                raise FileNotFoundError(
                    'Catalogue manifest is required to verify the shortened '
                    f'fingerprint for {run["name"]}.'
                )
            manifest = json.loads(
                CATALOGUE_MANIFEST_PATH.read_text(encoding='utf-8')
            )
            current_hash = manifest.get('catalogue_id_fingerprint_sha256')
            current_count = manifest.get('union_rows')
            if not current_hash or not current_hash.startswith(str(short_hash)):
                raise ValueError(
                    f"{run['name']} shortened catalogue fingerprint does not "
                    'match the current catalogue.'
                )
            declared_count = eligibility.get('catalogue_articles')
            if (
                declared_count is not None
                and current_count is not None
                and int(declared_count) != int(current_count)
            ):
                raise ValueError(
                    f"{run['name']} run_config declares {declared_count} "
                    f'catalogue articles; current catalogue has {current_count}.'
                )
            config['catalogue_id_fingerprint_sha256'] = current_hash
            needs_normalization = True
        else:
            warnings.warn(
                f"{run['name']} {split}: run_config has no catalogue ID "
                'fingerprint; skipping run_config verification for this model.'
            )
            return None

    declared_split = config.get('split')
    scope_split = config.get('scope', {}).get('split')
    declared_split = declared_split or scope_split
    if declared_split and declared_split != split:
        raise ValueError(
            f"{run['name']} run_config is for {declared_split}, not {split}. "
            'Supply the correct split-specific config or omit run_config.'
        )

    if needs_normalization:
        policy = eligibility.get('policy')
        if policy:
            config['pool_policy'] = {'name': policy}
        config['split'] = split
        exports = config.get('exports', {}).get(split, {})
        scope = config.get('scope', {})
        export_count = (
            exports.get('requests')
            or scope.get('export_requests')
        )
        if export_count is not None:
            config['export_request_scope'] = {
                'request_count': int(export_count)
            }
        normalized_path = (
            output_dir / f"{run['name'].lower().replace(' ', '_')}"
            '_evaluator_run_config.json'
        )
        normalized_path.write_text(
            json.dumps(config, indent=2) + '\n',
            encoding='utf-8',
        )
        return normalized_path
    return source


for split in SPLITS:
    labels_path = LABELS_BY_SPLIT[split]
    output_dir = STAGE1_OUTPUT_ROOT / SPLIT_OUTPUT_NAMES[split]
    output_dir.mkdir(parents=True, exist_ok=True)
    model_runs = model_runs_for_split(split)

    split_required = [labels_path]
    split_required.extend(Path(run['rankings']) for run in model_runs)
    missing = [str(path) for path in split_required if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            f'{split} input(s) missing:\n' + '\n'.join(missing)
        )

    print(f'\n===== Evaluating {SPLIT_OUTPUT_NAMES[split]} split =====')
    for run in model_runs:
        ranking_path = Path(run['rankings'])
        stem = run['name'].lower().replace(' ', '_')
        report_path = output_dir / f'{stem}_report.json'
        per_request_path = output_dir / f'{stem}_per_request.parquet'

        args = [
            'evaluate.py',
            '--split', split,
            '--labels', str(labels_path),
            '--articles', str(ARTICLES_PATH),
            '--rankings', str(ranking_path),
            '--export-k', str(EXPORT_K),
            '--ks', *[str(k) for k in KS],
            '--output', str(report_path),
            '--per-request-output', str(per_request_path),
        ]

        run_config = evaluator_compatible_run_config(
            run, split, output_dir
        )
        optional_inputs = [
            ('catalogue_scores', '--catalogue-scores'),
            ('recorded_slate_scores', '--recorded-slate-scores'),
        ]
        if run_config is not None:
            args.extend(['--run-config', str(run_config)])
        for key, flag in optional_inputs:
            value = run.get(key)
            if value is not None:
                value = Path(value)
                if not value.is_file():
                    raise FileNotFoundError(
                        f"{run['name']} {split} {key} not found: {value}"
                    )
                args.extend([flag, str(value)])

        if ALLOW_SUBSET:
            args.append('--allow-subset')

        old_argv = sys.argv
        try:
            sys.argv = args
            print(f"Evaluating {run['name']} on {SPLIT_OUTPUT_NAMES[split]}...")
            with redirect_stdout(io.StringIO()):
                evaluate_main()
        finally:
            sys.argv = old_argv

        report = json.loads(report_path.read_text(encoding='utf-8'))
        if report.get('split') != split:
            raise RuntimeError(
                f'{run["name"]}: evaluator reported split '
                f'{report.get("split")!r}, expected {split!r}'
            )
        pool_status = (
            'run config verified'
            if 'run_config' in report.get('inputs', {})
            else 'run config unavailable; ranking eligibility checked'
        )
        print(f"Done: {run['name']} on {SPLIT_OUTPUT_NAMES[split]} ({pool_status}).")

    print(f'{SPLIT_OUTPUT_NAMES[split]} evaluation complete: {output_dir}')


===== Evaluating training split =====
Evaluating LSTM on training...
Done: LSTM on training (run config unavailable; ranking eligibility checked).
Evaluating XGBoost on training...
Done: XGBoost on training (run config unavailable; ranking eligibility checked).
Evaluating Gemini on training...
Done: Gemini on training (run config unavailable; ranking eligibility checked).
training evaluation complete: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 1/training

===== Evaluating validation split =====
Evaluating LSTM on validation...


/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/code/evaluate.py:1243: UserWarning: Run config has no catalogue content fingerprint; metadata identity is verified only by article IDs
  warnings.warn("Run config has no catalogue content fingerprint; metadata identity is verified only by article IDs")
/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/code/evaluate.py:1248: UserWarning: Run config has no split field; split identity is taken from --split and the labels
  warnings.warn("Run config has no split field; split identity is taken from --split and the labels")


Done: LSTM on validation (run config verified).
Evaluating XGBoost on validation...


/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/code/evaluate.py:1243: UserWarning: Run config has no catalogue content fingerprint; metadata identity is verified only by article IDs
  warnings.warn("Run config has no catalogue content fingerprint; metadata identity is verified only by article IDs")
/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/code/evaluate.py:1257: UserWarning: Run config has no export_request_scope; exact scope is recorded from evaluated rankings
  warnings.warn("Run config has no export_request_scope; exact scope is recorded from evaluated rankings")


Done: XGBoost on validation (run config verified).


/tmp/ipykernel_1192/1812353670.py:64: UserWarning: Gemini validation: run_config has no catalogue ID fingerprint; skipping run_config verification for this model.
  warnings.warn(


Evaluating Gemini on validation...
Done: Gemini on validation (run config unavailable; ranking eligibility checked).
Evaluating RRF_lstm75_xgboost25 on validation...
Done: RRF_lstm75_xgboost25 on validation (run config unavailable; ranking eligibility checked).
validation evaluation complete: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 1/validation

===== Evaluating test split =====
Evaluating LSTM on test...
Done: LSTM on test (run config unavailable; ranking eligibility checked).
Evaluating XGBoost on test...
Done: XGBoost on test (run config unavailable; ranking eligibility checked).
Evaluating Gemini on test...
Done: Gemini on test (run config unavailable; ranking eligibility checked).
Evaluating RRF_lstm75_xgboost25 on test...
Done: RRF_lstm75_xgboost25 on test (run config unavailable; ranking eligibility checked).
test evaluation complete: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 1/test


## Verify eligible-catalogue counts by split

This checks each split’s saved per-request outputs against the shared catalogue’s publication timestamps. Training, validation, and test inputs and outputs remain separate; each is matched only with its own labels and rankings.

In [13]:
published_ns = (
    pd.to_datetime(
        pd.read_parquet(ARTICLES_PATH, columns=['published_time'])['published_time'],
        utc=True,
    )
    .dropna()
    .sort_values()
    .astype('datetime64[ns, UTC]')
    .astype('int64')
    .to_numpy()
)

for split in SPLITS:
    split_output_dir = STAGE1_OUTPUT_ROOT / SPLIT_OUTPUT_NAMES[split]
    for run in model_runs_for_split(split):
        stem = run['name'].lower().replace(' ', '_')
        per_request_path = (
            split_output_dir / f'{stem}_per_request.parquet'
        )
        if not per_request_path.is_file():
            print(f'{SPLIT_OUTPUT_NAMES[split]} {run["name"]}: per-request output missing')
            continue

        per = pd.read_parquet(
            per_request_path,
            columns=['impression_time', 'eligible_catalogue_count'],
        )
        times_ns = (
            pd.to_datetime(per['impression_time'], utc=True)
            .astype('datetime64[ns, UTC]')
            .astype('int64')
            .to_numpy()
        )
        expected_counts = np.searchsorted(
            published_ns,
            times_ns,
            side='right',
        )
        actual_counts = per['eligible_catalogue_count'].to_numpy(
            dtype='int64'
        )
        mismatched = actual_counts != expected_counts
        if mismatched.any():
            display(per.loc[mismatched].head(10))
            raise ValueError(
                f'{run["name"]} {split}: eligible-pool check failed for '
                f'{int(mismatched.sum())} requests'
            )
        print(
            f'{SPLIT_OUTPUT_NAMES[split]} {run["name"]}: eligible-pool check passed '
            f'({len(per):,} requests).'
        )

training LSTM: eligible-pool check passed (232,887 requests).
training XGBoost: eligible-pool check passed (232,887 requests).
training Gemini: eligible-pool check passed (232,887 requests).
validation LSTM: eligible-pool check passed (29,110 requests).
validation XGBoost: eligible-pool check passed (29,110 requests).
validation Gemini: eligible-pool check passed (29,110 requests).
validation RRF_lstm75_xgboost25: eligible-pool check passed (29,110 requests).
test LSTM: eligible-pool check passed (29,142 requests).
test XGBoost: eligible-pool check passed (29,142 requests).
test Gemini: eligible-pool check passed (29,142 requests).
test RRF_lstm75_xgboost25: eligible-pool check passed (29,142 requests).


In [14]:
import json
import pandas as pd

for split in SPLITS:
    split_name = SPLIT_OUTPUT_NAMES[split]
    report_dir = STAGE1_OUTPUT_ROOT / split_name
    report_paths = []
    for run in model_runs_for_split(split):
        stem = run['name'].lower().replace(' ', '_')
        report_path = report_dir / f"{stem}_report.json"
        if not report_path.is_file():
            raise FileNotFoundError(
                f"Expected report missing for {split_name}/{run['name']}: {report_path}"
            )
        report_paths.append(report_path)

    rows = []
    for report_path in report_paths:
        report = json.loads(report_path.read_text(encoding="utf-8"))

        # Prevent a report from accidentally appearing in the wrong split's CSV.
        if report.get("split") != split:
            raise ValueError(
                f"{report_path} is marked {report.get('split')!r}, "
                f"but is in the {split_name!r} folder."
            )

        metrics = pd.json_normalize(
            report.get("metrics", {}),
            sep=".",
        ).to_dict(orient="records")[0]
        slate_report = report.get("recorded_slate_diagnostics")
        slate_metrics = (
            pd.json_normalize(slate_report, sep=".").to_dict(orient="records")[0]
            if slate_report else {}
        )
        metrics.update({
            f"recorded_slate.{key}": value
            for key, value in slate_metrics.items()
        })

        rows.append({
            "model": report_path.stem.removesuffix("_report"),
            "split": split_name,
            "requests": report.get("evaluated_request_count"),
            **metrics,
        })

    summary = pd.DataFrame(rows)

    # Put common relevance metrics first when they exist; retain every other metric too.
    first_columns = [
        col for col in [
            "model",
            "split",
            "requests",
            "known_click_recall@100",
            "observed_click_ndcg@100",
            "known_click_recall@200",
            "observed_click_ndcg@200",
        ]
        if col in summary.columns
    ]
    remaining_columns = [
        col for col in summary.columns if col not in first_columns
    ]
    summary = summary[first_columns + remaining_columns]

    output_path = report_dir / "all_model_metrics_summary.csv"
    summary.to_csv(output_path, index=False)

    print(f"{split_name.title()} metrics saved to: {output_path}")
    display(summary)

Training metrics saved to: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 1/training/all_model_metrics_summary.csv


,model,split,requests,known_click_recall@100,observed_click_ndcg@100,known_click_recall@200,observed_click_ndcg@200,label_requests_available,requests_without_export,requests_with_known_clicks,...,click_exclusions.missing_publication_time,click_exclusions.published_after_request,recorded_slate.scope,recorded_slate.impressions,recorded_slate.candidate_rows,recorded_slate.mean_impression_auc,recorded_slate.auc_eligible_impressions,recorded_slate.auc_excluded_single_class,recorded_slate.mean_impression_ndcg@5,recorded_slate.ndcg@5_excluded_no_click
0,lstm,training,232887,0.946712,0.339213,0.965163,0.341805,232887,0,232887,...,0,41,recorded_slate_diagnostic,232887.0,2585747.0,0.68202,232887.0,0.0,0.506958,0.0
1,xgboost,training,232887,0.898183,0.231187,0.956098,0.239466,232887,0,232887,...,0,41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gemini,training,232887,0.030532,0.006253,0.045851,0.008391,232887,0,232887,...,0,41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Validation metrics saved to: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 1/validation/all_model_metrics_summary.csv


,model,split,requests,known_click_recall@100,observed_click_ndcg@100,known_click_recall@200,observed_click_ndcg@200,label_requests_available,requests_without_export,requests_with_known_clicks,...,click_exclusions.missing_publication_time,click_exclusions.published_after_request,recorded_slate.scope,recorded_slate.impressions,recorded_slate.candidate_rows,recorded_slate.mean_impression_auc,recorded_slate.auc_eligible_impressions,recorded_slate.auc_excluded_single_class,recorded_slate.mean_impression_ndcg@5,recorded_slate.ndcg@5_excluded_no_click
0,lstm,validation,29110,0.916103,0.286070,0.947656,0.290513,29110,0,29110,...,0,0,recorded_slate_diagnostic,29110.0,344666.0,0.613205,29110.0,0.0,0.435327,0.0
1,xgboost,validation,29110,0.858227,0.221182,0.922665,0.230362,29110,0,29110,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gemini,validation,29110,0.023033,0.004935,0.036345,0.006788,29110,0,29110,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,rrf_lstm75_xgboost25,validation,29110,0.914283,0.307936,0.947656,0.312644,29110,0,29110,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Test metrics saved to: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 1/test/all_model_metrics_summary.csv


,model,split,requests,known_click_recall@100,observed_click_ndcg@100,known_click_recall@200,observed_click_ndcg@200,label_requests_available,requests_without_export,requests_with_known_clicks,...,click_exclusions.missing_publication_time,click_exclusions.published_after_request,recorded_slate.scope,recorded_slate.impressions,recorded_slate.candidate_rows,recorded_slate.mean_impression_auc,recorded_slate.auc_eligible_impressions,recorded_slate.auc_excluded_single_class,recorded_slate.mean_impression_ndcg@5,recorded_slate.ndcg@5_excluded_no_click
0,lstm,test,29142,0.916706,0.288209,0.946507,0.292400,29142,0,29142,...,0,0,recorded_slate_diagnostic,29142.0,348951.0,0.60616,29142.0,0.0,0.42466,0.0
1,xgboost,test,29142,0.858251,0.221980,0.922174,0.231109,29142,0,29142,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gemini,test,29142,0.022806,0.004837,0.035880,0.006662,29142,0,29142,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,rrf_lstm75_xgboost25,test,29142,0.914956,0.311234,0.946439,0.315666,29142,0,29142,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Separation and usage notes

- Training labels and rankings are evaluated in-sample and written to Evaluation/Stage 1/training.
- Validation labels and rankings are used for selection and written to Evaluation/Stage 1/validation.
- Test labels and rankings are used for final reporting and written to Evaluation/Stage 1/test.
- The shared catalogue is built once from the declared article snapshots and used by both splits under the same eligibility policy.
- Split-specific run configs are optional. A config from one split is never reused for another split.
- Freeze model and reranker choices before interpreting the final test metrics. Do not tune using test results.
- Set ALLOW_SUBSET = True only for a declared sampled run with exact matching request IDs.